# 07 — Audio Assembly

**Purpose:** Place TTS segments at their original source timestamps, mix with the instrumental, and export the final dubbed audio.

## What this does

1. Load `instrumental.wav` at its native sample rate (48 kHz typical)
2. Resample each TTS segment from 44 100 Hz → source SR using `soxr_hq` (upsampling only — non-destructive)
3. Place each segment at its original timestamp — no stretching; duration fit is handled upstream by the TTS loop
4. Sidechain duck: lower instrumental −9 dB under active speech (prevents vocal bleed-through from stem separation residue)
5. Mix and export

## Why SR resampling is non-negotiable
ElevenLabs outputs 44 100 Hz PCM. Source video is typically 48 000 Hz. Mixing without resampling causes a subtle pitch shift on the TTS (the frame rate doesn't match). Upsampling the TTS to source SR is non-destructive — no quality loss.

## Sidechain ducking
Stem separation is not perfect; the instrumental track retains a faint residue of the original voice (typically 10–20%). Ducking the instrumental by −9 dB during speech windows reduces this bleed-through and keeps the dubbed voice intelligible.

## LUFS normalization (optional)
Included as an optional step. ElevenLabs output is consistent, so the mix is usually close to a reasonable level without it. Run it if the output sounds too loud or quiet vs. the source.

**Input:** `tts_output/tts_manifest.json` · `stems/instrumental.wav` · `input/dub_original.mp4`
**Output:** `assembled/dubbed_audio.wav` · `assembled/final_dubbed.mp4` · `results/final_output.mp4`

In [ ]:
!pip install -q pyloudnorm soundfile librosa soxr tqdm
print('Ready.')


In [ ]:
import sys, os, json, subprocess, time
os.environ['PATH'] = '/opt/homebrew/bin:' + os.environ.get('PATH', '')
import numpy as np
import soundfile as sf
import librosa
from tqdm.notebook import tqdm
from IPython.display import Audio, display

sys.path.insert(0, os.path.abspath('..'))
from config import (
    STEMS_DIR, TTS_OUTPUT_DIR, ASSEMBLED_DIR, RESULTS_DIR,
    SOURCE_VIDEO_PATH, TTS_OUTPUT_SR,
    TARGET_LUFS, INSTR_GAIN_NORMAL, INSTR_GAIN_DUCKED,
    load_source_sr,
)

MANIFEST_JSON = os.path.join(TTS_OUTPUT_DIR, 'tts_manifest.json')
INSTR_WAV     = os.path.join(STEMS_DIR,      'instrumental.wav')
OUT_AUDIO     = os.path.join(ASSEMBLED_DIR,  'dubbed_audio.wav')
OUT_VIDEO     = os.path.join(ASSEMBLED_DIR,  'final_dubbed.mp4')
FINAL_VIDEO   = os.path.join(RESULTS_DIR,    'final_output.mp4')

with open(MANIFEST_JSON) as f:
    manifest = json.load(f)
print(f'Loaded {len(manifest)} TTS segments')

SOURCE_SR = load_source_sr()
print(f'Source SR: {SOURCE_SR} Hz  /  TTS SR: {TTS_OUTPUT_SR} Hz')
if SOURCE_SR != TTS_OUTPUT_SR:
    print(f'Will upsample TTS {TTS_OUTPUT_SR} -> {SOURCE_SR} Hz at mix time')

In [ ]:
# ── Load instrumental at NATIVE SR ────────────────────────────────────────────
print(f'Loading instrumental at {SOURCE_SR} Hz...')
y_instr, _ = librosa.load(INSTR_WAV, sr=SOURCE_SR, mono=False, res_type='soxr_hq')
if y_instr.ndim == 1:
    y_instr = np.stack([y_instr, y_instr])
TOTAL_SAMPLES  = y_instr.shape[1]
TOTAL_DURATION = TOTAL_SAMPLES / SOURCE_SR
print(f'Instrumental: shape={y_instr.shape}  dur={TOTAL_DURATION:.2f}s')

In [ ]:
# ── Build dialogue track & speech mask ───────────────────────────────────────
y_dialogue  = np.zeros_like(y_instr)
speech_mask = np.zeros(TOTAL_SAMPLES, dtype=np.float32)

overruns     = []
placement_log = []

for item in tqdm(manifest, desc='Placing TTS segments'):
    seg_path = item['path']
    if not os.path.exists(seg_path):
        print(f'WARNING: missing {seg_path}')
        continue

    y_tts_orig, _ = librosa.load(seg_path, sr=TTS_OUTPUT_SR, mono=False, res_type='soxr_hq')
    if y_tts_orig.ndim == 1:
        y_tts_orig = np.stack([y_tts_orig, y_tts_orig])

    # Resample TTS to source SR (upsampling 44100→48000; non-destructive)
    if SOURCE_SR != TTS_OUTPUT_SR:
        y_tts = librosa.resample(y_tts_orig, orig_sr=TTS_OUTPUT_SR, target_sr=SOURCE_SR, res_type='soxr_hq')
    else:
        y_tts = y_tts_orig

    start_sample   = int(item['start'] * SOURCE_SR)
    target_samples = int(item['target_duration'] * SOURCE_SR)

    # Log if segment runs over its slot — duration fix should happen upstream in TTS loop
    if y_tts.shape[1] > target_samples * 1.12:
        overruns.append({
            'index':      item['index'],
            'actual_dur': item['actual_duration'],
            'target_dur': item['target_duration'],
            'ratio':      item['duration_ratio'],
        })

    end_sample = start_sample + y_tts.shape[1]
    if end_sample > TOTAL_SAMPLES:
        y_tts      = y_tts[:, :TOTAL_SAMPLES - start_sample]
        end_sample = TOTAL_SAMPLES

    y_dialogue[:, start_sample:end_sample] += y_tts
    speech_mask[start_sample:end_sample]    = 1.0

    placement_log.append({
        'index':      item['index'],
        'start':      item['start'],
        'end':        item['end'],
        'actual_dur': item['actual_duration'],
        'target_dur': item['target_duration'],
        'ratio':      item['duration_ratio'],
    })

print(f'Placed {len(placement_log)} segments.')
if overruns:
    print(f'Overruns (>12% long, re-run TTS for these): {len(overruns)}')
    for o in overruns[:10]:
        print(f'  [{o["index"]:04d}] ratio={o["ratio"]:.2f}  actual={o["actual_dur"]:.2f}s  target={o["target_dur"]:.2f}s')

In [ ]:
# ── Sidechain ducking: instrumental ducks under dialogue ─────────────────────
# Stem separation leaves 10-20% vocal residue in the instrumental track.
# Dropping the instrumental -9 dB during speech reduces bleed-through and
# keeps the dubbed voice clearly intelligible over background music.

# Smooth the speech mask (10 ms fade at boundaries) to avoid clicks
from scipy.ndimage import uniform_filter1d
fade_samples = int(0.010 * SOURCE_SR)
smooth_mask  = uniform_filter1d(speech_mask.astype(np.float64), size=fade_samples).astype(np.float32)

duck_gain      = INSTR_GAIN_NORMAL * (1.0 - smooth_mask) + INSTR_GAIN_DUCKED * smooth_mask
y_instr_ducked = y_instr[:, :TOTAL_SAMPLES] * duck_gain[np.newaxis, :]
print(f'Sidechain ducking: {INSTR_GAIN_NORMAL:.2f} → {INSTR_GAIN_DUCKED:.2f} under speech')

In [ ]:
# ── Mix ───────────────────────────────────────────────────────────────────────
y_mix = y_dialogue + y_instr_ducked

# Clip prevention
peak = float(np.max(np.abs(y_mix)))
if peak > 0.99:
    y_mix = y_mix / peak * 0.99
    print(f'Peak {peak:.3f} > 0.99 — normalized to prevent clipping')

print(f'Mix: shape={y_mix.shape}  dur={y_mix.shape[1]/SOURCE_SR:.2f}s  at {SOURCE_SR} Hz')

In [ ]:
# ── LUFS normalization — OPTIONAL ─────────────────────────────────────────────
# ElevenLabs output is consistent and the instrumental preserves original levels,
# so the mix is usually at a reasonable level without this step.
# Run this cell if the output sounds noticeably louder or quieter than the source.
# Target: -16 LUFS (EBU R128 / Netflix / Amazon Prime delivery spec).
try:
    import pyloudnorm as pyln

    meter   = pyln.Meter(SOURCE_SR)
    y_mix_T = y_mix.T  # pyloudnorm expects (samples, channels)
    lufs_in = meter.integrated_loudness(y_mix_T)
    print(f'Loudness before: {lufs_in:.1f} LUFS  (target {TARGET_LUFS})')

    y_norm_T   = pyln.normalize.loudness(y_mix_T, lufs_in, TARGET_LUFS)
    y_norm     = y_norm_T.T
    lufs_after = meter.integrated_loudness(y_norm_T)
    true_peak  = 20 * np.log10(np.max(np.abs(y_norm)) + 1e-12)
    print(f'Loudness after:  {lufs_after:.1f} LUFS')
    print(f'True peak:       {true_peak:.1f} dBFS')
    if true_peak > -1.0:
        print('WARNING: true peak > -1.0 dBTP — apply a limiter for broadcast delivery')

except Exception as e:
    print(f'pyloudnorm skipped ({e}) — using mix as-is')
    y_norm     = y_mix
    lufs_after = None

In [ ]:
y_out = y_norm.T  # soundfile: (samples, channels)
sf.write(OUT_AUDIO, y_out, SOURCE_SR, subtype='PCM_16')
print(f'Saved: {OUT_AUDIO}  ({os.path.getsize(OUT_AUDIO)/1e6:.1f} MB  at {SOURCE_SR} Hz)')


In [ ]:
# ── Mux audio back to video ───────────────────────────────────────────────────
for out_path in [OUT_VIDEO, FINAL_VIDEO]:
    cmd = [
        'ffmpeg', '-y',
        '-i', SOURCE_VIDEO_PATH,
        '-i', OUT_AUDIO,
        '-c:v', 'copy',
        '-c:a', 'aac', '-b:a', '256k',
        '-ar', str(SOURCE_SR),  # preserve source SR in video container
        '-map', '0:v:0', '-map', '1:a:0',
        '-shortest', out_path
    ]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(f'ffmpeg error: {res.stderr[-1500:]}')
    else:
        print(f'Saved: {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)')


In [ ]:
import pandas as pd
df = pd.DataFrame(placement_log)
print('=== Assembly metrics ===')
print(f'Segments placed:  {len(df)}')
print(f'Overruns (>12%):  {len(overruns)}  (these should be re-generated in notebook 06)')
print(f'Output LUFS:      {lufs_after}')
print(f'Source SR:        {SOURCE_SR} Hz')
print(f'Output:           {FINAL_VIDEO}')
print()
print(df[['index','start','end','actual_dur','target_dur','ratio']].to_string(index=False))

In [ ]:
from config import AUDIO_EXTRACTED_DIR
print('Dubbed audio:')
display(Audio(OUT_AUDIO))
print('\nOriginal audio (A/B comparison):')
display(Audio(os.path.join(AUDIO_EXTRACTED_DIR, 'source_audio.wav')))